In [0]:
bronze_path = "abfss://bronze@stnyc311.dfs.core.windows.net/nyc311/nyc311_raw.json"

df_bronze = spark.read.json(bronze_path)

display(df_bronze.limit(5))

In [0]:
df_bronze.printSchema()

In [0]:
total_rows = df_bronze.count()
unique_keys = df_bronze.select("unique_key").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Unique keys: {unique_keys}")
print(f"Duplicate rows by unique_key: {total_rows - unique_keys}")

In [0]:
from pyspark.sql import functions as F

null_counts = df_bronze.select([
    F.sum(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
])

display(null_counts)

In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    .withColumn(
        "created_date",
        F.to_timestamp("created_date", "yyyy-MM-dd'T'HH:mm:ss.SSS")
    )
    .withColumn(
        "closed_date",
        F.to_timestamp("closed_date", "yyyy-MM-dd'T'HH:mm:ss.SSS")
    )
    .withColumn("borough", F.upper(F.trim("borough")))
    .withColumn("status", F.trim("status"))
    .withColumn("complaint_type", F.trim("complaint_type"))
    .withColumn("descriptor", F.trim("descriptor"))
    .withColumn("agency", F.trim("agency"))
    .withColumn("agency_name", F.trim("agency_name"))
    .withColumn("incident_zip", F.trim("incident_zip"))
    .withColumn("location_type", F.trim("location_type"))
    .withColumn("open_data_channel_type", F.trim("open_data_channel_type"))
)

df_silver.printSchema()

In [0]:
df_silver.select(
    F.sum(F.when(F.col("created_date").isNull(), 1).otherwise(0)).alias("missing_created_date"),
    F.sum(F.when(F.col("closed_date").isNull(), 1).otherwise(0)).alias("missing_closed_date")
).show()

In [0]:
invalid_resolution_dates = df_silver.filter(
    F.col("closed_date").isNotNull() &
    (F.col("closed_date") < F.col("created_date"))
).count()

print(f"Closed before created: {invalid_resolution_dates}")

In [0]:
display(
    df_silver
    .filter(
        F.col("closed_date").isNotNull() &
        (F.col("closed_date") < F.col("created_date"))
    )
    .select(
        "unique_key",
        "created_date",
        "closed_date",
        "status",
        "agency",
        "complaint_type"
    )
    .limit(20)
)

In [0]:
invalid_diffs = (
    df_silver
    .filter(
        F.col("closed_date").isNotNull() &
        (F.col("closed_date") < F.col("created_date"))
    )
    .withColumn(
        "diff_seconds",
        F.col("closed_date").cast("long") - F.col("created_date").cast("long")
    )
)

invalid_diffs.select(
    F.min("diff_seconds").alias("most_negative_seconds"),
    F.max("diff_seconds").alias("least_negative_seconds")
).show()

In [0]:
invalid_diffs.select(
    F.sum(F.when(F.col("diff_seconds") >= -60, 1).otherwise(0)).alias("within_60_seconds"),
    F.sum(F.when((F.col("diff_seconds") < -60) & (F.col("diff_seconds") >= -3600), 1).otherwise(0)).alias("between_1min_1hr"),
    F.sum(F.when((F.col("diff_seconds") < -3600) & (F.col("diff_seconds") >= -86400), 1).otherwise(0)).alias("between_1hr_1day"),
    F.sum(F.when(F.col("diff_seconds") < -86400, 1).otherwise(0)).alias("over_1day")
).show()

In [0]:
display(
    invalid_diffs
    .filter(F.col("diff_seconds") < -60)
    .select(
        "unique_key",
        "created_date",
        "closed_date",
        "diff_seconds",
        "status",
        "agency",
        "complaint_type"
    )
    .orderBy("diff_seconds")
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "resolution_seconds",
        F.when(
            F.col("closed_date").isNotNull(),
            F.col("closed_date").cast("long") - F.col("created_date").cast("long")
        )
    )
    .withColumn(
        "resolution_hours",
        F.when(F.col("closed_date").isNull(), F.lit(None).cast("double"))
         .when(F.col("resolution_seconds") < -60, F.lit(None).cast("double"))
         .when(F.col("resolution_seconds") < 0, F.lit(0.0))
         .otherwise(F.col("resolution_seconds") / 3600.0)
    )
    .withColumn(
        "resolution_time_flag",
        F.when(F.col("closed_date").isNull(), "OPEN_OR_UNCLOSED")
         .when(F.col("resolution_seconds") < -60, "INVALID_NEGATIVE")
         .when(F.col("resolution_seconds") < 0, "ADJUSTED_TO_ZERO")
         .otherwise("VALID")
    )
)

In [0]:
df_silver.groupBy("resolution_time_flag").count().show()

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stnyc311.dfs.core.windows.net/nyc311/")

In [0]:
df_silver_check = spark.read.format("delta").load(
    "abfss://silver@stnyc311.dfs.core.windows.net/nyc311/"
)

df_silver_check.count()

In [0]:
df_silver_check.groupBy("resolution_time_flag").count().show()

In [0]:
df_silver_check \
    .groupBy("complaint_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10, truncate=False)